# NB04 – O que a média esconde
**Métodos e Ferramentas de Data Science – FGV MBA · Aula 4**

## A história
A **Horizonte Distribuidora** (empresa fictícia do NB01 e do trabalho em grupo) vai começar a vender pela internet, direto para o consumidor. Antes de definir quanto cobrar de frete, a diretoria pediu uma referência à Marina, gerente de frota, que agora também cuida das entregas. Chega um e-mail dela:

> *"Achei uma base pública de um e-commerce brasileiro, com mais de 112 mil itens vendidos. Preciso de duas respostas: **quanto custa um frete típico?** E **que cobranças fogem do padrão e merecem revisão?** Só não quero levar para a diretoria um número puxado por meia dúzia de casos estranhos."*

Ela tem razão em se preocupar. Na NB01a, a cidade de maior renda média só liderava por causa de **um** cliente. Por isso, antes de responder, você vai fazer o que todo analista faz primeiro: **olhar a base e tratá-la**. Só depois vai **descrever** o frete.

| Seções | O que você vai fazer |
|---|---|
| 1 a 7 · **Tratar antes de medir** | faltantes, valores impossíveis, tipos, linhas repetidas, unidades e faixas |
| 8 a 13 · **Descrever a distribuição** | tipo de variável, histograma, média × mediana, dispersão, percentis, box plot e outliers |
| 14 · **Sua vez** | exercícios individuais |
| 15 · **Para fechar** | a resposta à Marina |

## Os dados
**Olist**, e-commerce brasileiro, pedidos de 2016 a 2018: [Brazilian E-Commerce Public Dataset (Kaggle)](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce).
Para a aula, os itens, os produtos, os pedidos e os clientes já vêm **juntos num arquivo só**, sem nenhum valor alterado: os defeitos que você vai encontrar estão na base original. Cada linha é **um item vendido**.

## Como usar
Execute as células **de cima para baixo**, uma por vez, com **Shift + Enter**. Leia o resultado de cada uma antes de seguir.

Antes de cada célula de código há um bloco **"O que acontece na célula abaixo"**: leia primeiro, execute depois e confira se o resultado bate com o que você esperava.

Nas **Questões** ao longo do caminho, escreva uma linha de código na célula vazia logo abaixo. A dica mostra qual comando usar.

# Tratar antes de medir

## 1. Carregar a base

**O que acontece na célula abaixo**

- Os três `import` você já conhece do NB01: `pandas` (tabelas), `matplotlib` e `seaborn` (gráficos).
- `import numpy as np` carrega o **NumPy**, a biblioteca de contas do Python, uma das que você conheceu na Aula 3. Hoje vamos usar dele só uma coisa: o `np.nan`, que representa um valor **faltante** (seção 4).
- `pd.set_option(...)` mostra os números com **2 casas decimais**. Muda só a forma de mostrar, não o valor guardado.

**Resultado esperado:** nenhuma saída. Se não aparecer erro, as ferramentas foram carregadas.

In [ ]:
import pandas as pd               # tabelas
import numpy as np                # contas; np.nan = valor faltante
import matplotlib.pyplot as plt   # gráficos
import seaborn as sns             # gráficos com visual pronto

pd.set_option("display.float_format", "{:.2f}".format)   # mostra números com 2 casas

**O que acontece na célula abaixo**

- `URL = "..."` guarda o endereço do arquivo; `pd.read_csv(URL)` baixa o arquivo e cria a tabela `df`, como no NB01. O `.csv.gz` é um CSV compactado; o pandas descompacta sozinho.
- `print(df.shape)` mostra o tamanho da tabela: **(linhas, colunas)**.
- `df.head()` mostra as 5 primeiras linhas.

> **Deu erro de conexão?** Veja o *Plano B* no fim do notebook.

**Resultado esperado:** `(112650, 14)` e uma tabela com 5 linhas. Repare que, nas três primeiras, a data de entrega (`order_delivered_customer_date`) aparece como `NaN`: é assim que o pandas mostra um valor **faltante**.

In [ ]:
URL = "https://raw.githubusercontent.com/carloscarlim/fgv-mfds/main/dados/olist_itens_aula04.csv.gz"
df = pd.read_csv(URL)
print(df.shape)   # (linhas, colunas)
df.head()

**O que cada coluna guarda** (os nomes estão em inglês, como na base original)

| Coluna | O que é |
|---|---|
| `order_id`, `order_item_id` | código do pedido e posição do item dentro dele (1, 2, 3...) |
| `order_status` | situação do pedido: `delivered` (entregue), `shipped` (enviado), `canceled` (cancelado) etc. |
| `customer_state` | estado (UF) do cliente |
| `order_purchase_timestamp` | data e hora da compra |
| `order_delivered_customer_date` | data em que o cliente recebeu |
| `order_estimated_delivery_date` | data de entrega prometida |
| `product_category_name` | categoria do produto |
| `product_weight_g`, `product_length_cm`, `product_height_cm`, `product_width_cm` | peso (gramas) e medidas (centímetros) do produto |
| `price`, `freight_value` | preço do item e frete do item, em R\$ |

Um pedido com 3 produtos aparece em **3 linhas**. Guarde isso: vai importar na seção 6.

## 2. Olhar antes de tratar
**Parte 1: a base chegou.** 112 mil linhas, e a vontade é calcular a média do frete e responder logo. Mas a primeira lição da Aula 3 vale aqui: **antes da conta, olhe a tabela.** Duas perguntas: de que tipo é cada coluna? E onde faltam valores?

**O que acontece na célula abaixo**

- `df.info()` mostra, para cada coluna, **quantos valores estão preenchidos** (`Non-Null Count`) e o **tipo** (`Dtype`). Você viu esse comando na NB01a.
- Tipos que aparecem aqui: `int64` (número inteiro), `float64` (número com casas decimais) e `object` (**texto**).

**Resultado esperado:** `112650 entries` e 14 colunas. A data de entrega tem só `110196 non-null`, a categoria `111047`, o peso e as medidas `112632`. E as três colunas de data aparecem como `object`: estão guardadas como **texto**.

In [ ]:
df.info()

**Como ler:**
- `Non-Null Count` menor que o número de linhas = há valores **faltantes** naquela coluna.
- `Dtype` `object` = **texto**. As três datas vieram como texto: do jeito que estão, não dá para subtrair uma da outra (seção 5).

**O que acontece na célula abaixo**

- `df.isna()` pergunta, célula a célula, "este valor está faltando?" e responde com `True` ou `False`.
- `.sum()` soma cada coluna. Como `True` vale 1 e `False` vale 0, a soma é **quantos faltam** em cada coluna. É a mesma linha da cola da Aula 3.

**Resultado esperado:** 2454 itens sem data de entrega, 1603 sem categoria e 18 sem peso e medidas. As outras colunas têm 0.

In [ ]:
df.isna().sum()   # quantos valores faltam em cada coluna

**Questão 1.** Que **porcentagem** dos itens está sem categoria (`product_category_name`)?
Dica: `df["product_category_name"].isna()` devolve `True` onde falta. A **média** de uma coluna de `True`/`False` é a proporção de `True`; use `.mean()`.

In [ ]:
# Questão 1: escreva aqui


## 3. Faltante é informação
**Parte 2: os 2.454 itens sem entrega.** A tentação é preencher a data que falta com a média e seguir em frente. Antes, uma pergunta: **por que** ela falta?

O que fazer com um valor que falta? As opções mais comuns:

| Opção | Quando faz sentido |
|---|---|
| Descartar a **linha** | poucas linhas, e o faltante não tem padrão |
| Descartar a **coluna** | a coluna está quase toda vazia |
| **Substituir** pela média, mediana ou valor mais frequente | o faltante é raro e não muda a conclusão |
| Criar uma **categoria própria** (`"sem_categoria"`) | variável de texto: mantém a linha nas contagens e deixa o faltante visível |
| **Manter** e explicar | o faltante tem um motivo e é a resposta certa |

**O que acontece na célula abaixo**

- `sem_entrega = df["order_delivered_customer_date"].isna()` guarda uma coluna de `True`/`False`: `True` nos itens **sem** data de entrega.
- `sem_entrega.sum()` conta os `True`: quantos itens estão sem data.
- `df.loc[sem_entrega, "order_status"]` usa o `.loc[linha, coluna]` que você viu no NB01, com uma diferença: no lugar do nome da linha vai a coluna de `True`/`False`. O pandas fica só com as linhas `True` e com a coluna `order_status`. É um **filtro** e uma **seleção de coluna** ao mesmo tempo.
- `.value_counts()` conta quantos itens há de cada status.

**Resultado esperado:** `Itens sem data de entrega: 2454` e a contagem por status: `shipped` (enviado) 1185, `canceled` (cancelado) 535, `invoiced` (faturado) 359, `processing` (em processamento) 357, `delivered` (entregue) 8, `unavailable` (indisponível) 7 e `approved` (aprovado) 3.

In [ ]:
sem_entrega = df["order_delivered_customer_date"].isna()
print("Itens sem data de entrega:", sem_entrega.sum())
df.loc[sem_entrega, "order_status"].value_counts()   # o status desses itens

Quase todos os itens sem data de entrega são de pedidos que **ainda não foram entregues** ou foram **cancelados**. Para eles, o faltante **é a resposta certa**: preencher com a média inventaria uma entrega que não aconteceu.

Só os 8 itens com status `delivered` e sem data são, de fato, **erro de registro**.

Já a **categoria** faltante é diferente: o produto foi vendido, só não tem categoria no cadastro. Aqui criamos uma categoria própria.

**O que acontece na célula abaixo**

- `.fillna("sem_categoria")` **preenche** os valores faltantes da coluna com o texto `"sem_categoria"`. Os valores que já existiam não mudam.
- `df["product_category_name"] = ...` guarda o resultado **na mesma coluna**, por cima da antiga. Por isso esta célula muda a tabela: se você executá-la de novo, nada acontece (não há mais faltantes para preencher).
- `.value_counts().head(8)` mostra as 8 categorias com mais itens.

**Resultado esperado:** `cama_mesa_banho` em primeiro, com 11115 itens, seguida de `beleza_saude` (9670) e `esporte_lazer` (8641). A `sem_categoria`, com 1603 itens, não entra entre as 8 maiores, mas agora conta como uma categoria.

In [ ]:
df["product_category_name"] = df["product_category_name"].fillna("sem_categoria")
df["product_category_name"].value_counts().head(8)

## 4. Valores impossíveis
Um valor pode estar preenchido e, mesmo assim, estar errado. Um produto que **pesa 0 grama** não existe.

**O que acontece na célula abaixo**

- `df["product_weight_g"] == 0` compara cada peso com zero e devolve `True`/`False`. Lembre: `==` **compara**; `=` **guarda**.
- `(...).sum()` conta os `True`: quantos itens têm peso zero.
- `df.loc[df["product_weight_g"] == 0, [...]]` fica só com esses itens e com as **quatro colunas** da lista. Uma lista de nomes entre colchetes seleciona várias colunas de uma vez.

**Resultado esperado:** `Itens com peso igual a zero: 8` e uma tabela com os 8: todos de `cama_mesa_banho`, com preço de R\$ 100,00 ou R\$ 129,90 e frete cobrado (de R\$ 14,49 a R\$ 23,85).

In [ ]:
print("Itens com peso igual a zero:", (df["product_weight_g"] == 0).sum())
df.loc[df["product_weight_g"] == 0, ["product_category_name", "product_weight_g", "price", "freight_value"]]

São produtos de cama, mesa e banho, com frete cobrado: o peso real existe, só não foi cadastrado. Trocamos o zero por **faltante**, para ele não entrar nas contas de peso.

**O que acontece na célula abaixo**

- `.replace(0, np.nan)` troca todo valor **0** da coluna por `np.nan`, o "valor faltante" do NumPy. O pandas passa a tratar esses pesos como os outros que faltam: eles ficam de fora das médias e das contagens.
- Por que não apagar as linhas? Porque o preço e o frete desses itens estão certos; só o peso está errado.
- `.isna().sum()` conta os pesos faltantes depois da troca.

**Resultado esperado:** `Itens sem peso válido: 26` (os 18 que já faltavam + os 8 zeros).

In [ ]:
df["product_weight_g"] = df["product_weight_g"].replace(0, np.nan)
print("Itens sem peso válido:", df["product_weight_g"].isna().sum())

## 5. Tipos: de texto para data
Com as datas no formato certo, o pandas sabe fazer contas com elas: quantos dias entre a compra e a entrega, por exemplo.

**O que acontece na célula abaixo**

- `for coluna in [...]:` repete o bloco de baixo para cada nome da lista, como o `for` da NB01a. Assim, uma linha só converte as três colunas.
- `pd.to_datetime(df[coluna])` lê o texto `"2016-09-15 12:16:38"` e o transforma numa **data de verdade**.
- `df[coluna] = ...` guarda a coluna convertida por cima da antiga.
- `df.dtypes` mostra o tipo de cada coluna (a mesma informação do `info()`, sem as contagens).

**Resultado esperado:** as três colunas de data agora aparecem como `datetime64[ns]` (data e hora). As outras não mudam.

In [ ]:
for coluna in ["order_purchase_timestamp", "order_delivered_customer_date", "order_estimated_delivery_date"]:
    df[coluna] = pd.to_datetime(df[coluna])

df.dtypes

**O que acontece na célula abaixo**

- Subtrair uma data de outra dá o **tempo entre elas** (por exemplo, `10 days 04:31:12`).
- `.dt.days` pega desse tempo só o número de **dias inteiros**. O `.dt` é a "caixa de ferramentas de datas" do pandas.
- `df["prazo_dias"] = ...` cria a coluna nova com o prazo de cada item.
- `.describe()` resume a coluna: quantos valores, média, desvio, mínimo, quartis e máximo.

**Resultado esperado:** `count` de 110196 (só os itens entregues), média de 12 dias, mediana (`50%`) de 10 dias, mínimo 0 e máximo de 209 dias.

In [ ]:
# Prazo de entrega: dias entre a compra e o recebimento
df["prazo_dias"] = (df["order_delivered_customer_date"] - df["order_purchase_timestamp"]).dt.days
df["prazo_dias"].describe()

Repare que o `count` do prazo é menor que o número de linhas: pedido sem entrega não tem prazo, e as contas do pandas **ignoram os faltantes** sozinhas.

**Questão 2.** Qual é o prazo **mediano** de entrega, em dias? E o **maior** prazo?
Dica: `df["prazo_dias"].median()` e `df["prazo_dias"].max()`.

In [ ]:
# Questão 2: escreva aqui


## 6. Linha repetida nem sempre é erro: item × pedido
**Parte 3: o número que quase foi para a diretoria.** A Marina pergunta, de passagem: *"quantos pedidos foram entregues?"*. Contar as linhas com `delivered` parece a resposta. Mas cada linha é um **item**: um pedido com 3 itens aparece 3 vezes.

**O que acontece na célula abaixo**

- `df.drop_duplicates(subset="order_id")` apaga as linhas **repetidas** pelo código do pedido e fica com **uma linha por pedido** (a primeira de cada um).
- `pedidos = ...` guarda o resultado numa tabela nova. A `df` continua inteira, com uma linha por item.
- `len(...)` conta as linhas de cada tabela.
- `(df["order_status"] == "delivered").sum()` conta as linhas com status entregue: primeiro na tabela de itens, depois na de pedidos.
- `print()` sozinho pula uma linha, só para separar os resultados.

**Resultado esperado:** 112650 itens e 98666 pedidos; pedidos entregues: **110197** contando linhas e **96478** contando pedidos.

In [ ]:
pedidos = df.drop_duplicates(subset="order_id")
print("Linhas (itens):    ", len(df))
print("Pedidos diferentes:", len(pedidos))
print()
print("Pedidos entregues, contando linhas: ", (df["order_status"] == "delivered").sum())
print("Pedidos entregues, contando pedidos:", (pedidos["order_status"] == "delivered").sum())

Contando linhas, a Marina levaria à diretoria **14% mais "pedidos entregues"** do que a empresa de fato entregou. A regra: **antes de contar, saiba o que cada linha representa.**

## 7. Padronizar unidades e criar faixas
**Padronizar** é colocar os valores numa unidade que facilite a comparação e a leitura. Gramas viram quilos; centímetros cúbicos viram litros.

**O que acontece na célula abaixo**

- `df["peso_kg"] = df["product_weight_g"] / 1000` cria uma coluna nova dividindo cada peso por 1.000, como no NB01: o pandas faz a conta para todas as linhas de uma vez.
- `volume_litros` multiplica comprimento × altura × largura (cm³) e divide por 1.000 (1 litro = 1.000 cm³).
- `df[["product_weight_g", "peso_kg", "volume_litros"]]` usa **dois pares de colchetes**: o de fora seleciona colunas, o de dentro é a **lista** com os nomes. O `.describe()` resume as três juntas.

**Resultado esperado:** peso mediano (`50%`) de 700 g, ou 0,70 kg; o maior pesa 40,42 kg; volume mediano de 6,48 litros. O `count` do peso é 112624: os 26 sem peso válido ficaram de fora.

In [ ]:
df["peso_kg"] = df["product_weight_g"] / 1000
df["volume_litros"] = df["product_length_cm"] * df["product_height_cm"] * df["product_width_cm"] / 1000
df[["product_weight_g", "peso_kg", "volume_litros"]].describe()

**Criar faixas** (discretizar) transforma um número contínuo em poucas categorias ordenadas. Fica mais fácil comparar grupos e explicar o resultado para a diretoria.

**O que acontece na célula abaixo**

- `pd.cut(...)` corta a coluna `peso_kg` em faixas.
- `bins=[0, 0.5, 2, 10, 50]` são os **limites**: de 0 a 0,5 kg, de 0,5 a 2, de 2 a 10 e de 10 a 50. Em Python, o decimal é com **ponto** (`0.5`).
- `labels=[...]` dá um **nome** a cada faixa, na mesma ordem. São 5 limites para 4 nomes.
- `.value_counts()` conta os itens de cada faixa; `.sort_index()` mostra as faixas **na ordem dos pesos**, e não da maior contagem para a menor.

**Resultado esperado:** até 0,5 kg: 47478 itens; 0,5 a 2 kg: 39810; 2 a 10 kg: 20040; acima de 10 kg: 5296. Os 26 itens sem peso ficam sem faixa.

In [ ]:
df["faixa_peso"] = pd.cut(df["peso_kg"],
                          bins=[0, 0.5, 2, 10, 50],
                          labels=["até 0,5 kg", "0,5 a 2 kg", "2 a 10 kg", "acima de 10 kg"])
df["faixa_peso"].value_counts().sort_index()

**Questão 3.** Crie a coluna `frete_por_kg` (frete dividido pelo peso em kg). Qual é a sua **mediana**?
Dica: crie a coluna como o `peso_kg` acima, dividindo `df["freight_value"]` por `df["peso_kg"]`; depois, `.median()`.

In [ ]:
# Questão 3: escreva aqui


# Descrever a distribuição

## 8. Que tipo de variável é?
A base está tratada, e cada decisão ficou escrita acima. Agora, sim, dá para descrever o frete. Antes de escolher um gráfico ou uma medida, classifique a variável:

| Classe | Subtipo | Exemplo nesta base | Medidas e gráficos |
|---|---|---|---|
| **Quantitativa** (número que se mede ou conta) | Contínua | `price`, `freight_value`, `peso_kg` | média, mediana, desvio; histograma, box plot |
| | Discreta | `order_item_id`, `prazo_dias` | contagens, média, mediana |
| **Qualitativa** (categoria) | Nominal (sem ordem) | `product_category_name`, `customer_state` | frequência; gráfico de barras |
| | Ordinal (com ordem) | `faixa_peso` (seção 7) | frequência, mediana |

**Cuidado:** `order_id` parece um número de controle, mas é um código. Calcular a "média do código" não faz sentido.

## 9. A forma da distribuição: o histograma
Um **histograma** divide os valores em faixas e mostra, em cada barra, **quantos itens** caem naquela faixa. É o mesmo tipo de gráfico do fim da NB01a.

**O que acontece na célula abaixo**

- `frete = df["freight_value"]` dá um nome curto à coluna do frete. Daqui em diante, `frete` é essa coluna.
- `fig, ax = plt.subplots(1, 2, figsize=(12, 4))` abre uma figura com **dois gráficos lado a lado** (1 linha, 2 colunas). O `ax[0]` é o da esquerda; o `ax[1]`, o da direita.
- `sns.histplot(frete, bins=80, ax=ax[0])` desenha o histograma do frete com 80 barras no gráfico da esquerda.
- No da direita, `frete[frete > 0]` filtra só os fretes maiores que zero, e `log_scale=True` usa a **escala logarítmica** no eixo horizontal (explicada abaixo).
- `.set_title(...)` dá um título a cada gráfico; `plt.show()` mostra a figura.

**Resultado esperado:** à esquerda, quase todas as barras encostadas no começo do eixo (fretes de R\$ 10 a R\$ 25) e uma cauda fina que vai até R\$ 400; à direita, a mesma informação, com a forma mais fácil de ver.

In [ ]:
frete = df["freight_value"]

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(frete, bins=80, ax=ax[0])
ax[0].set_title("Frete por item (R$)")
sns.histplot(frete[frete > 0], bins=80, log_scale=True, ax=ax[1])
ax[1].set_title("Frete por item, escala logarítmica")
plt.show()

A distribuição tem uma **cauda longa à direita**: a maioria dos fretes é baixa, e alguns poucos são muito altos.
Na escala logarítmica (cada marca vale 10 vezes a anterior), a forma fica mais fácil de ler. Essa escala não aceita o valor zero; por isso o segundo gráfico usa só os fretes maiores que zero.

## 10. Medidas de posição: média, mediana e moda
**Parte 4: o frete típico.** É a primeira pergunta da Marina.

**O que acontece na célula abaixo**

- `frete.mean()` calcula a **média**; `frete.median()`, a **mediana** (o valor do meio: metade dos fretes está abaixo dele, metade acima).
- `frete.mode()` devolve a **moda**, o valor mais frequente. Ela vem como uma lista, porque pode haver empate; o `[0]` pega o primeiro valor.
- As linhas `print(f"...")` são **f-strings**, como no NB01: o que está entre `{ }` é trocado pelo valor, e o `:.2f` mostra 2 casas decimais.

**Resultado esperado:** `Média: R$ 19.99`, `Mediana: R$ 16.26` e `Moda: R$ 15.10`. O Python escreve o decimal com ponto.

In [ ]:
print(f"Média:   R$ {frete.mean():.2f}")
print(f"Mediana: R$ {frete.median():.2f}   (metade dos fretes está abaixo deste valor)")
print(f"Moda:    R$ {frete.mode()[0]:.2f}   (o valor mais frequente)")

**Média maior que a mediana** é o sinal de uma cauda à direita: poucos valores altos puxam a média para cima.
Para dizer à Marina "quanto custa um frete típico", a **mediana** representa melhor o cliente comum.

## 11. Dispersão, percentis e assimetria
Saber o valor típico não basta: é preciso saber **o quanto os fretes variam** em torno dele.

**O que acontece na célula abaixo**

- `frete.quantile([0.25, 0.75])` calcula o **1º quartil** (Q1: 25% dos fretes estão abaixo dele) e o **3º quartil** (Q3: 75% estão abaixo). `q1, q3 = ...` guarda os dois valores, um em cada nome, na mesma linha.
- `iqr = q3 - q1` é a **amplitude interquartil**: o tamanho do "miolo" onde estão os 50% do meio.
- `frete.std()` é o **desvio padrão**: o quanto os valores se afastam da média, em reais.
- `frete.std() / frete.mean()` é o **coeficiente de variação**; o `:.0%` mostra o número como porcentagem, sem casas decimais.
- A última linha calcula vários **percentis** de uma vez: P10, P25, P50, P75, P90 e P99.

**Resultado esperado:** desvio padrão de R\$ 15.81, IQR de R\$ 8.07, coeficiente de variação de 79% e a tabela de percentis: 8.73 · 13.08 · 16.26 · 21.15 · 34.04 · 84.52.

In [ ]:
q1, q3 = frete.quantile([0.25, 0.75])
iqr = q3 - q1
print(f"Desvio padrão: R$ {frete.std():.2f}")
print(f"Amplitude interquartil (IQR = Q3 − Q1): R$ {iqr:.2f}   (o 'miolo' com 50% dos fretes)")
print(f"Coeficiente de variação: {frete.std() / frete.mean():.0%}   (desvio em relação à média)")
print()
frete.quantile([0.10, 0.25, 0.50, 0.75, 0.90, 0.99])

**O que acontece na célula abaixo**

- `frete.skew()` mede a **assimetria**: 0 numa distribuição simétrica; positiva quando a cauda vai para a direita; negativa quando vai para a esquerda.

**Resultado esperado:** `Assimetria: 5.64`, um número bem alto: a cauda à direita é longa.

In [ ]:
print(f"Assimetria: {frete.skew():.2f}   (0 = simétrica; positiva = cauda à direita)")

**Como ler:**
- **Percentil 90:** 90% dos fretes custam até esse valor. Percentis viram regras que a operação entende: "frete acima do P99 vai para revisão".
- **Desvio padrão × IQR:** o desvio sofre com os valores extremos; o IQR, não.
- **Assimetria** alta e positiva confirma a cauda à direita do histograma e a média acima da mediana (seção 10). Com uma cauda assim, a média e o desvio padrão são puxados pelos extremos; a mediana e o IQR, não.

## 12. Box plot: a distribuição em uma figura
A caixa vai do Q1 ao Q3; a linha do meio é a mediana; as hastes vão até 1,5 × IQR além da caixa. O que passa das hastes aparece como ponto.

Com a faixa de peso da seção 7, dá para comparar grupos:

**O que acontece na célula abaixo**

- `plt.figure(figsize=(9, 4))` abre uma figura de 9 por 4 polegadas.
- `sns.boxplot(data=df, x="faixa_peso", y="freight_value", ...)` desenha **um box plot para cada faixa de peso**: no eixo x, as faixas; no eixo y, o frete. O seaborn faz o agrupamento sozinho, como um `groupby`.
- `showfliers=False` esconde os pontos extremos. Com eles, as caixas ficariam espremidas no fundo do gráfico.
- `plt.title`, `plt.ylabel` e `plt.xlabel` dão título e nomes aos eixos (`""` deixa o eixo x sem nome). O `;` separa dois comandos na mesma linha.

**Resultado esperado:** quatro caixas que sobem da esquerda para a direita. A da faixa acima de 10 kg fica bem mais alta e mais larga que as outras.

In [ ]:
plt.figure(figsize=(9, 4))
sns.boxplot(data=df, x="faixa_peso", y="freight_value", showfliers=False)
plt.title("Frete por faixa de peso (sem os pontos extremos)")
plt.ylabel("Frete (R$)"); plt.xlabel("")
plt.show()

## 13. Outliers: erro, fraude ou exceção legítima?
**Parte 5: as cobranças que fogem do padrão.** É a segunda pergunta da Marina. Duas regras comuns para apontar valores fora do padrão:
- **Regra do IQR** (a do box plot): fora de [Q1 − 1,5 × IQR, Q3 + 1,5 × IQR].
- **Z-score:** mais de 3 desvios padrão longe da média. Pensada para distribuições parecidas com a normal; funciona mal com cauda longa, como a do frete.

**O que acontece na célula abaixo**

- `lim_sup = q3 + 1.5 * iqr` calcula o **limite superior** da regra do IQR, com o `q3` e o `iqr` da seção 11.
- `z = (frete - frete.mean()) / frete.std()` calcula o **z-score** de cada frete: quantos desvios padrão ele está longe da média.
- `(frete > lim_sup).sum()` conta os fretes acima do limite; `(frete > lim_sup).mean()` dá a **proporção** (a média de `True`/`False`, como na Questão 1), e o `:.1%` mostra em porcentagem com 1 casa.
- `z.abs()` é o valor **absoluto** (sem sinal): conta tanto quem está muito acima quanto muito abaixo da média.

**Resultado esperado:** `Limite superior pela regra do IQR: R$ 33.25`; `11613 itens (10.3%)` acima dele; `2041 itens (1.8%)` com z-score acima de 3.

In [ ]:
lim_sup = q3 + 1.5 * iqr
z = (frete - frete.mean()) / frete.std()
print(f"Limite superior pela regra do IQR: R$ {lim_sup:.2f}")
print(f"Acima do limite do IQR: {(frete > lim_sup).sum()} itens ({(frete > lim_sup).mean():.1%})")
print(f"Z-score acima de 3:     {(z.abs() > 3).sum()} itens ({(z.abs() > 3).mean():.1%})")

**O que acontece na célula abaixo**

- `df.sort_values("freight_value", ascending=False)` ordena a tabela pelo frete, do **maior** para o menor (`ascending=False` = ordem decrescente).
- `[[...]]` seleciona só as cinco colunas que ajudam a julgar cada caso.
- `.head(8)` mostra os 8 primeiros: os 8 fretes mais altos da base.

**Resultado esperado:** o maior frete é de R\$ 409,68, num produto da categoria `bebes` de 14,68 kg enviado ao PI. Vários dos seguintes são produtos de 25 a 30 kg, com preço acima de R\$ 900.

In [ ]:
# Os fretes mais altos: erro, fraude ou exceção legítima?
df.sort_values("freight_value", ascending=False)[["product_category_name", "peso_kg", "customer_state", "price", "freight_value"]].head(8)

**Pare e pense:** um frete de centenas de reais para um produto de 30 kg é erro? E para um produto de 200 g?
Outlier é uma **pergunta**, não uma resposta: a regra estatística diz **onde olhar**; o contexto diz **o que é**.

E depois de investigar? Duas saídas comuns, cada uma com um custo:
- **Remover** (*trimming*): tira as linhas da análise. Perde informação, e a base fica menor.
- **Limitar** (*capping*): troca o valor extremo pelo limite. Mantém as linhas, mas altera os valores.

**O que acontece na célula abaixo**

- `removendo = frete[frete <= lim_sup]` é um **filtro**: fica só com os fretes até o limite. Os outros saem.
- `limitando = frete.clip(upper=lim_sup)` mantém todos os fretes, mas troca os que passam do limite **pelo próprio limite** (R\$ 33,25).
- `pd.DataFrame({...})` monta uma tabela com três colunas, uma para cada versão, lado a lado. Dentro das chaves, cada par `"nome": valores` vira uma coluna; os valores são o `.describe()` de cada versão.

**Resultado esperado:** removendo, sobram 101037 itens e a média cai para 16.04; limitando, ficam os 112650 itens e a média vai para 17.82. A mediana muda pouco: 15.58 removendo e 16.26 limitando (a mesma do original).

In [ ]:
removendo = frete[frete <= lim_sup]           # remover (trimming)
limitando = frete.clip(upper=lim_sup)          # limitar (capping)

pd.DataFrame({"original": frete.describe(),
              "removendo": removendo.describe(),
              "limitando": limitando.describe()})

Remover tirou cerca de 10% dos itens; limitar manteve todos, mas "achatou" os fretes altos. A **média** caiu de cerca de R\$ 20 para R\$ 16 (removendo) ou R\$ 18 (limitando); a **mediana** mudou bem menos. **Para descrever o frete típico, a mediana já resiste aos extremos sem apagar nada.**
Nenhuma das saídas é a certa sempre: depende da pergunta. E toda decisão de tratamento precisa ficar **registrada**, porque muda os números.

---
## 14. Sua vez: exercícios individuais
Os comandos de que você precisa já apareceram acima. A dica de cada exercício diz em que seção.

**Exercício 1.** Calcule a média, a mediana e a assimetria do **preço** (`price`). A distribuição é simétrica?
Dica: `.mean()`, `.median()` e `.skew()`, como nas seções 10 e 11, agora com `df["price"]`.

In [ ]:
# SEU CÓDIGO AQUI


**Exercício 2.** Qual é o **percentil 99** do preço? Quantos itens ficam acima dele?
Dica: `.quantile(0.99)` dá o percentil (seção 11); para contar quem passa dele, compare a coluna com o valor e some os `True` (seção 13).

In [ ]:
# SEU CÓDIGO AQUI


**Exercício 3.** Calcule a **mediana do frete** em cada faixa de peso. Quantas vezes o frete mediano de um item acima de 10 kg é maior que o de um item até 0,5 kg?
Dica: `df.groupby("faixa_peso", observed=True)["freight_value"].median()`. O `observed=True` mostra só as faixas que existem na base.

In [ ]:
# SEU CÓDIGO AQUI


**Exercício 4.** Use a tabela `pedidos` (uma linha por pedido, seção 6). Calcule o **limite superior da regra do IQR** para o prazo de entrega (`prazo_dias`). Quantos pedidos passam desse limite? Você os removeria da análise?
Dica: repita as contas das seções 11 e 13 com `pedidos["prazo_dias"]` no lugar de `frete`.

In [ ]:
# SEU CÓDIGO AQUI


**Desafio.** Quantos itens têm **frete maior que o preço** do produto? Que porcentagem isso representa? É erro, fraude ou pode ser legítimo?
Dica: `df["freight_value"] > df["price"]` compara as duas colunas, linha a linha.

In [ ]:
# SEU CÓDIGO AQUI


---
## 15. Para fechar: a resposta à Marina (e o que ela não diz)
- **Frete típico:** a mediana (cerca de R\$ 16) representa melhor o cliente comum do que a média (cerca de R\$ 20).
- **Fora do padrão:** pela regra do IQR, cerca de 10% dos itens têm frete acima de R\$ 33. É muito para revisar um a um: vale começar por uma faixa mais extrema (por exemplo, acima do P99, cerca de R\$ 85) e olhar o peso de cada item.
- **Tratamentos feitos e registrados:** categoria faltante virou `"sem_categoria"`; peso zero virou faltante; itens sem entrega ficaram sem prazo; contagens de pedidos usam uma linha por pedido.

**Limitações:** a base não tem a distância da entrega (CEP), que também pesa no frete; preço e frete são por item, não por pedido; os dados são de 2016 a 2018, de outra empresa: servem de referência para a Horizonte, não de tabela de preço.

**Referências:** box plot e regra do IQR: Tukey, J. W. *Exploratory Data Analysis*. Addison-Wesley, 1977. Remover e limitar outliers (*trimming* e *capping*): Galli, S. *Python Feature Engineering Cookbook*. Packt, 2020.

## Erros comuns e como resolver

| Mensagem | O que aconteceu | O que fazer |
|---|---|---|
| `NameError: name 'df' is not defined` | A célula da seção 1 não foi executada | Execute as células desde o início (*Ambiente de execução → Executar anteriores*) |
| `NameError: name 'frete'` (ou `'q3'`, `'lim_sup'`) `is not defined` | A célula que cria esse nome não foi executada | Execute as células anteriores |
| `KeyError: 'faixa_peso'` ou `'prazo_dias'` | A seção que cria a coluna não foi executada | Execute as células anteriores |
| `KeyError: 'Price'` | Nome da coluna com letra diferente | Os nomes são em minúsculas e em inglês: `"price"` |
| `TypeError: unsupported operand type(s) for -: 'str' and 'str'` | As datas ainda estão como texto | Execute a célula do `pd.to_datetime` (seção 5) |
| Aviso `FutureWarning: ... observed=False` | Mudança de comportamento do pandas | Pode ignorar, ou use `observed=True` no `groupby` |
| Gráfico em branco | Faltou `plt.show()` ou a célula foi interrompida | Execute de novo |
| Nada acontece / célula com `[*]` | A célula ainda está rodando | Aguarde alguns segundos |

## Plano B: se o download falhar
1. Baixe o arquivo `olist_itens_aula04.csv.gz` da pasta `dados/` do repositório da disciplina (ou pelo link do ECLASS).
2. No Colab, clique no ícone de **pasta** na barra lateral esquerda e arraste o arquivo para lá.
3. Execute a célula abaixo **no lugar da segunda célula da seção 1** e siga normalmente.

**O que acontece na célula abaixo**

- As linhas começam com `#`, então estão **comentadas**: do jeito que está, a célula não faz nada.
- Para usar o Plano B, apague o `# ` do começo das três linhas e execute. O `pd.read_csv` passa a ler o arquivo da pasta do Colab, em vez de baixá-lo da internet.
- Depois, siga normalmente a partir da seção 2.

In [ ]:
# PLANO B: só execute se o download da seção 1 falhou
# df = pd.read_csv("olist_itens_aula04.csv.gz")
# print(df.shape)
# df.head()

### Autor e licença

**Autor:** Prof. Carlos Carlim · [LinkedIn](https://www.linkedin.com/in/carloscarlim-77144120) · [GitHub](https://github.com/carloscarlim)

**Licença MIT.** Copyright (c) 2026 Carlos Carlim. Os direitos autorais deste material pertencem exclusivamente ao autor.

É permitido usar, copiar, modificar, mesclar, publicar, distribuir, sublicenciar e vender cópias deste material, desde que este aviso de copyright e o texto da licença sejam mantidos em todas as cópias ou partes substanciais. O material é fornecido "no estado em que se encontra", sem garantia de qualquer tipo. Texto completo da licença: [LICENSE](https://github.com/carloscarlim/fgv-mfds/blob/main/LICENSE).

Os dados usados nos notebooks têm licenças próprias, indicadas em [`dados/README.md`](https://github.com/carloscarlim/fgv-mfds/blob/main/dados/README.md).